In [1]:
from pyspark.sql import SparkSession

In [2]:
spark = SparkSession.builder.appName('Logistic').getOrCreate()

In [4]:
data=spark.read.csv("/content/drive/MyDrive/titanic.csv",inferSchema=True,header=True)

In [5]:
data.printSchema()

root
 |-- PassengerId: integer (nullable = true)
 |-- Survived: integer (nullable = true)
 |-- Pclass: integer (nullable = true)
 |-- Name: string (nullable = true)
 |-- Sex: string (nullable = true)
 |-- Age: double (nullable = true)
 |-- SibSp: integer (nullable = true)
 |-- Parch: integer (nullable = true)
 |-- Ticket: string (nullable = true)
 |-- Fare: double (nullable = true)
 |-- Cabin: string (nullable = true)
 |-- Embarked: string (nullable = true)



In [6]:
data.columns

['PassengerId',
 'Survived',
 'Pclass',
 'Name',
 'Sex',
 'Age',
 'SibSp',
 'Parch',
 'Ticket',
 'Fare',
 'Cabin',
 'Embarked']

In [7]:
Data=data.select(['Survived',
 'Pclass','Sex',
 'Age',
 'SibSp',
 'Parch','Fare', 'Embarked'
])

In [8]:
Final_Data=Data.na.drop()

In [9]:
from pyspark.ml.feature import (VectorAssembler,VectorIndexer,OneHotEncoder,StringIndexer)

In [10]:
Sex_Indexer=StringIndexer(inputCol="Sex",outputCol="Sex_Index");
Sex_Encoder=OneHotEncoder(inputCol="Sex_Index",outputCol="Sex_Vec");

In [12]:
Emabrk_Indexer=StringIndexer(inputCol="Embarked",outputCol="Embarked_Index");
Emabrk_Encoder=OneHotEncoder(inputCol="Embarked_Index",outputCol="Embarked_Vec");

In [11]:
assembler=VectorAssembler(inputCols=[
    'Embarked_Vec',
 'Pclass','Sex_Vec',
 'Age',
 'SibSp',
 'Parch','Fare'
],outputCol="features")

In [13]:
from pyspark.ml.classification import LogisticRegression

In [14]:
from pyspark.ml import Pipeline

In [15]:
lr_model=LogisticRegression(featuresCol='features',labelCol='Survived')

In [16]:
pipeline=Pipeline(stages=[
    Sex_Indexer,Emabrk_Indexer,Sex_Encoder,Emabrk_Encoder,assembler,lr_model
])

In [17]:
train_data,test_data = Final_Data.randomSplit([0.7,0.3])

In [18]:
model=pipeline.fit(train_data)

In [19]:
results=model.transform(test_data)

In [20]:
results.select(['Survived','prediction']).show()

+--------+----------+
|Survived|prediction|
+--------+----------+
|       0|       1.0|
|       0|       1.0|
|       0|       1.0|
|       0|       1.0|
|       0|       0.0|
|       0|       1.0|
|       0|       0.0|
|       0|       0.0|
|       0|       0.0|
|       0|       0.0|
|       0|       0.0|
|       0|       0.0|
|       0|       0.0|
|       0|       0.0|
|       0|       0.0|
|       0|       0.0|
|       0|       0.0|
|       0|       0.0|
|       0|       0.0|
|       0|       0.0|
+--------+----------+
only showing top 20 rows
